In [1]:
 !pip install -q transformers accelerate faiss-cpu

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.3 MB/s eta 0:00:00


In [17]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "google/gemma-2-2b"
tok = AutoTokenizer.from_pretrained(MODEL_ID)

# Dynamic device allocation to support CPU fallback when CUDA is not available
device = "cuda" if torch.cuda.is_available() else "cpu"

# If using CPU, we use float32 to avoid precision/support issues with float16
torch_dtype = torch.float16 if device == "cuda" else torch.float32

model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch_dtype).to(device).eval()

def embed(texts: list[str]) -> np.ndarray:
    batch = tok(texts, return_tensors="pt", padding=True,
                truncation=True, max_length=256).to(device)
    with torch.no_grad():
        hidden = model(**batch).last_hidden_state        # (B, T, D)
    mask = batch["attention_mask"].unsqueeze(-1)          # (B, T, 1)
    pooled = (hidden * mask).sum(1) / mask.sum(1)         # mean pooling
    vec = torch.nn.functional.normalize(pooled.float(), dim=-1)
    return vec.cpu().numpy()

print(embed(["hello world"]).shape)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

(1, 2304)


In [16]:
!pip install -q transformers accelerate faiss-cpu


In [4]:
import os
os.makedirs("app", exist_ok=True)
open("app/__init__.py", "w").close()

In [18]:
%%writefile app/detection.py
import re

RULES = {
    "CREDIT_CARD": re.compile(r"\b(?:\d[ -]?){12,18}\d\b"),
    "EMAIL": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "SSN":   re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "PHONE": re.compile(r"(?<!\d)\+?\d[\d\s-]{8,}\d(?!\d)"),
    "IPV4":  re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
}

def luhn_valid(number: str) -> bool:
    digits = [int(c) for c in number if c.isdigit()]
    if not 13 <= len(digits) <= 19:
        return False
    total = 0
    for i, d in enumerate(reversed(digits)):
        if i % 2 == 1:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0

def detect(text: str) -> list[tuple[str, int, int]]:
    out = []
    for label, pat in RULES.items():
        for m in pat.finditer(text):
            if label == "CREDIT_CARD" and not luhn_valid(m.group(0)):
                continue
            out.append((label, m.start(), m.end()))

    out.sort(key=lambda x: (x[1], -(x[2] - x[1])))
    result, last_end = [], -1
    for f in out:
        if f[1] >= last_end:
            result.append(f)
            last_end = f[2]
    return result

def sanitize(text: str) -> str:
    for label, s, e in sorted(detect(text), key=lambda x: x[1], reverse=True):
        text = text[:s] + f"<{label}>" + text[e:]
    return text

Overwriting app/detection.py


In [6]:
import sys
import os
import importlib

# Force Python to prefer the correct directory paths
for path in ['/content/gemma-embeddings-practice', '/content', os.getcwd()]:
    if path in sys.path:
        sys.path.remove(path)
    sys.path.insert(0, path)

# Unload cached module if it was partially or incorrectly loaded
if 'app' in sys.modules:
    sys.modules.pop('app', None)
if 'app.detection' in sys.modules:
    sys.modules.pop('app.detection', None)

# Import the fresh implementation
from app.detection import detect, sanitize

sample = "Card 4111 1111 1111 1111 se payment, email a@b.com, SSN 123-45-6789"
print("Detected PII:", detect(sample))
print("Sanitized Text:", sanitize(sample))

Detected PII: [('CREDIT_CARD', 5, 24), ('EMAIL', 43, 50), ('SSN', 56, 67)]
Sanitized Text: Card <CREDIT_CARD> se payment, email <EMAIL>, SSN <SSN>


In [7]:
import faiss
from app.detection import sanitize

sentences = [
    "Mera email jane@example.com hai",
    "Mujhe apna phone number 98765 43210 par call karo",
    "Card 4111 1111 1111 1111 se payment hua",
    "Aaj mausam bahut achha hai",
    "Main cricket khelna pasand karta hoon",
]

# Pehle sanitize, phir embed (privacy rule)
sanitized = [sanitize(s) for s in sentences]
vecs = embed(sanitized)

index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

query = "payment ka record kaise dekhun"
q = embed([sanitize(query)])
scores, ids = index.search(q, 3)

for i, s in zip(ids[0], scores[0]):
    print(round(float(s), 3), sanitized[i])

0.895 Main cricket khelna pasand karta hoon
0.886 Mujhe apna phone number <PHONE> par call karo
0.856 Aaj mausam bahut achha hai


In [ ]:
queries = [
    ("kisi ka phone number kya hai", 1),
    ("email bhejna hai", 0),
    ("credit card se paisa diya", 2),
    ("kal ka weather kaisa rahega", 3),
    ("cricket ka match", 4),
]

hits = 0
for q, expected in queries:
    qv = embed([sanitize(q)])
    scores, ids = index.search(qv, 3)
    top = list(ids[0])
    ok = expected in top
    hits += ok
    print("✅" if ok else "❌", q, "->", sanitized[top[0]])

print("Top-3 accuracy:", hits / len(queries))

In [ ]:
for q, expected in queries:
    qv = embed([sanitize(q)])
    scores, ids = index.search(qv, 3)
    print("\nQuery:", q)
    for i, s in zip(ids[0], scores[0]):
        print("  ", round(float(s), 3), sanitized[i])

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_st(texts):
    return st_model.encode(texts, normalize_embeddings=True)

vecs2 = embed_st(sanitized)
index2 = faiss.IndexFlatIP(vecs2.shape[1])
index2.add(vecs2)

hits2 = 0
for q, expected in queries:
    qv = embed_st([sanitize(q)])
    scores, ids = index2.search(qv, 3)
    top = list(ids[0])
    ok = expected in top
    hits2 += ok
    print("✅" if ok else "❌", q, "->", sanitized[top[0]], round(float(scores[0][0]), 3))

print("MiniLM Top-3 accuracy:", hits2 / len(queries))

In [ ]:
!ls -R app

In [13]:
%cd /content/gemma-embeddings-practice
!git pull origin main

[Errno 2] No such file or directory: '/content/gemma-embeddings-practice'
/content
fatal: not a git repository (or any of the parent directories): .git


In [ ]:
%cd /content
!rm -rf gemma-embeddings-practice
!git clone https://github.com/YOUR_USERNAME/gemma-embeddings-practice.git

In [ ]:
import os
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError, NotebookAccessError

try:
    # Trying to get GITHUB_TOKEN from Colab Secrets
    GH_TOKEN = userdata.get("GITHUB_TOKEN")
    USER = "Muskan-Savita"
    REPO = "gemma-embeddings-practice"

    %cd /content
    !rm -rf gemma-embeddings-practice
    !git clone https://{GH_TOKEN}@github.com/{USER}/{REPO}.git
except NotebookAccessError:
    print("❌ Access Error: 'GITHUB_TOKEN' exists but this notebook does not have access to it.")
    print("Please go to the Secrets panel (🔑 key icon on the left sidebar) and turn ON the toggle switch for 'GITHUB_TOKEN' to grant Notebook access.")
except SecretNotFoundError:
    print("❌ Error: 'GITHUB_TOKEN' was not found in your Colab Secrets.")
    print("Please add 'GITHUB_TOKEN' to your Secrets panel (🔑 key icon on the left sidebar) and enable Notebook access.")


In [ ]:
%cd /content/gemma-embeddings-practice
!cp -r /content/app .
!git status

In [ ]:
!git add app
!git commit -m "Add PII detection module"
!git push origin main

In [ ]:
!git config --global user.email "savitamuskan1404@gmail.com"
!git config --global user.name "Muskan-Savita"
!git commit -m "Add PII detection module"
!git push origin main

In [ ]:
%cd /content/gemma-embeddings-practice
!git config --global user.email "savitamuskan1404@gmail.com"
!git config --global user.name "Muskan-Savita"
!git add app
!git commit -m "Add PII detection module"
!git push origin main

In [ ]:
import os
from google.colab import userdata

GH_TOKEN = userdata.get("GITHUB_TOKEN")
USER = "Muskan-Savita"
REPO = "gemma-embeddings-practice"

%cd /content
!rm -rf gemma-embeddings-practice
!git clone https://{GH_TOKEN}@github.com/{USER}/{REPO}.git
%cd /content/gemma-embeddings-practice
!git config --global user.email "savitamuskan1404@gmail.com"
!git config --global user.name "Muskan-Savita"
!mkdir -p app
!touch app/__init__.py

In [ ]:
%%writefile /content/gemma-embeddings-practice/app/detection.py
import re

RULES = {
    "CREDIT_CARD": re.compile(r"\b(?:\d[ -]?){12,18}\d\b"),
    "EMAIL": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "SSN":   re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "PHONE": re.compile(r"(?<!\d)\+?\d[\d\s-]{8,}\d(?!\d)"),
    "IPV4":  re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
}

def luhn_valid(number: str) -> bool:
    digits = [int(c) for c in number if c.isdigit()]
    if not 13 <= len(digits) <= 19:
        return False
    total = 0
    for i, d in enumerate(reversed(digits)):
        if i % 2 == 1:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0

def detect(text: str) -> list[tuple[str, int, int]]:
    out = []
    for label, pat in RULES.items():
        for m in pat.finditer(text):
            if label == "CREDIT_CARD" and not luhn_valid(m.group(0)):
                continue
            out.append((label, m.start(), m.end()))

    out.sort(key=lambda x: (x[1], -(x[2] - x[1])))
    result, last_end = [], -1
    for f in out:
        if f[1] >= last_end:
            result.append(f)
            last_end = f[2]
    return result

def sanitize(text: str) -> str:
    for label, s, e in sorted(detect(text), key=lambda x: x[1], reverse=True):
        text = text[:s] + f"<{label}>" + text[e:]
    return text

In [ ]:
%cd /content/gemma-embeddings-practice
!git add app
!git commit -m "Add PII detection module"
!git push origin main

In [ ]:
import json, os

os.makedirs("data", exist_ok=True)

dataset = {
    "documents": [
        {"id": 0, "text": "Mera email jane@example.com hai", "pii": ["EMAIL"]},
        {"id": 1, "text": "Mujhe apna phone number 98765 43210 par call karo", "pii": ["PHONE"]},
        {"id": 2, "text": "Card 4111 1111 1111 1111 se payment hua", "pii": ["CREDIT_CARD"]},
        {"id": 3, "text": "Aaj mausam bahut achha hai", "pii": []},
        {"id": 4, "text": "Main cricket khelna pasand karta hoon", "pii": []},
        {"id": 5, "text": "Mera SSN 123-45-6789 hai", "pii": ["SSN"]},
        {"id": 6, "text": "Server ka IP 192.168.1.10 hai", "pii": ["IPV4"]},
        {"id": 7, "text": "Mera naya email priya.sharma@gmail.com hai", "pii": ["EMAIL"]},
        {"id": 8, "text": "Office ka number 011 4567 8901 hai", "pii": ["PHONE"]},
        {"id": 9, "text": "Hamara sales target is quarter mein badhaya gaya", "pii": []},
        {"id": 10, "text": "Naya laptop kharidna hai", "pii": []},
        {"id": 11, "text": "Credit card 5500 0000 0000 0004 se bill bhara", "pii": ["CREDIT_CARD"]},
        {"id": 12, "text": "Khana bahut tasty tha", "pii": []},
        {"id": 13, "text": "Meeting kal subah 10 baje hai", "pii": []},
        {"id": 14, "text": "Mera backup email rahul.k@yahoo.in hai", "pii": ["EMAIL"]},
        {"id": 15, "text": "Ghar ka router IP 10.0.0.1 hai", "pii": ["IPV4"]},
        {"id": 16, "text": "Dhoop aaj bahut tez hai", "pii": []},
        {"id": 17, "text": "Mujhe SSN update karna hai 987-65-4321", "pii": ["SSN"]},
        {"id": 18, "text": "Football match dekhna pasand hai", "pii": []},
        {"id": 19, "text": "Kal naya project shuru karna hai", "pii": []}
    ],
    "queries": [
        {"query": "kisi ka phone number kya hai", "expected": 1},
        {"query": "email bhejna hai", "expected": 0},
        {"query": "payment ka record kya hai", "expected": 2},
        {"query": "kal ka weather kaisa rahega", "expected": 3},
        {"query": "cricket ka match", "expected": 4},
        {"query": "server ka address kya hai", "expected": 6},
        {"query": "bank card ki details", "expected": 11},
        {"query": "sales ka target kya tha", "expected": 9},
        {"query": "football ke baare mein batao", "expected": 18},
        {"query": "kal ka plan kya hai", "expected": 19}
    ]
}

with open("data/labeled_sentences.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print("Saved:", len(dataset["documents"]), "docs,", len(dataset["queries"]), "queries")

In [ ]:
pip install -q faiss-cpu

In [ ]:
import os
from google.colab import userdata

try:
    # Try to clone git repo if it doesn't exist
    if not os.path.exists('/content/gemma-embeddings-practice'):
        GH_TOKEN = userdata.get("GITHUB_TOKEN")
        USER = "Muskan-Savita"
        REPO = "gemma-embeddings-practice"
        !git clone https://{GH_TOKEN}@github.com/{USER}/{REPO}.git

    # Sync app folder to root /content so 'import app' always works
    if os.path.exists('/content/gemma-embeddings-practice/app'):
        !cp -r /content/gemma-embeddings-practice/app /content/
        print("Successfully restored 'app' module!")
except Exception as e:
    print("Could not restore app folder automatically:", e)

In [ ]:
import sys
import os
# Ensure the directories containing 'app' are in python path
sys.path.extend(['/content/gemma-embeddings-practice', '/content', os.getcwd()])

import json, faiss, numpy as np
from app.detection import sanitize

# Dynamic fallback to ensure the file is always loaded correctly
path_primary = "/content/data/labeled_sentences.json"
path_secondary = "/content/gemma-embeddings-practice/data/labeled_sentences.json"

target_path = None
if os.path.exists(path_primary):
    target_path = path_primary
elif os.path.exists(path_secondary):
    target_path = path_secondary
else:
    target_path = "/content/gemma-embeddings-practice/data/labeled_sentences.json"

# Automatically re-create the data if it is missing
if not os.path.exists(target_path):
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    dataset = {
        "documents": [
            {"id": 0, "text": "Mera email jane@example.com hai", "pii": ["EMAIL"]},
            {"id": 1, "text": "Mujhe apna phone number 98765 43210 par call karo", "pii": ["PHONE"]},
            {"id": 2, "text": "Card 4111 1111 1111 1111 se payment hua", "pii": ["CREDIT_CARD"]},
            {"id": 3, "text": "Aaj mausam bahut achha hai", "pii": []},
            {"id": 4, "text": "Main cricket khelna pasand karta hoon", "pii": []},
            {"id": 5, "text": "Mera SSN 123-45-6789 hai", "pii": ["SSN"]},
            {"id": 6, "text": "Server ka IP 192.168.1.10 hai", "pii": ["IPV4"]},
            {"id": 7, "text": "Mera naya email priya.sharma@gmail.com hai", "pii": ["EMAIL"]},
            {"id": 8, "text": "Office ka number 011 4567 8901 hai", "pii": ["PHONE"]},
            {"id": 9, "text": "Hamara sales target is quarter mein badhaya gaya", "pii": []},
            {"id": 10, "text": "Naya laptop kharidna hai", "pii": []},
            {"id": 11, "text": "Credit card 5500 0000 0000 0004 se bill bhara", "pii": ["CREDIT_CARD"]},
            {"id": 12, "text": "Khana bahut tasty tha", "pii": []},
            {"id": 13, "text": "Meeting kal subah 10 baje hai", "pii": []},
            {"id": 14, "text": "Mera backup email rahul.k@yahoo.in hai", "pii": ["EMAIL"]},
            {"id": 15, "text": "Ghar ka router IP 10.0.0.1 hai", "pii": ["IPV4"]},
            {"id": 16, "text": "Dhoop aaj bahut tez hai", "pii": []},
            {"id": 17, "text": "Mujhe SSN update karna hai 987-65-4321", "pii": ["SSN"]},
            {"id": 18, "text": "Football match dekhna pasand hai", "pii": []},
            {"id": 19, "text": "Kal naya project shuru karna hai", "pii": []}
        ],
        "queries": [
            {"query": "kisi ka phone number kya hai", "expected": 1},
            {"query": "email bhejna hai", "expected": 0},
            {"query": "payment ka record kya hai", "expected": 2},
            {"query": "kal ka weather kaisa rahega", "expected": 3},
            {"query": "cricket ka match", "expected": 4},
            {"query": "server ka address kya hai", "expected": 6},
            {"query": "bank card ki details", "expected": 11},
            {"query": "sales ka target kya tha", "expected": 9},
            {"query": "football ke baare mein batao", "expected": 18},
            {"query": "kal ka plan kya hai", "expected": 19}
        ]
    }
    with open(target_path, "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

with open(target_path, encoding="utf-8") as f:
    data = json.load(f)

docs = [d["text"] for d in data["documents"]]
sanitized = [sanitize(s) for s in docs]
queries = data["queries"]

def top3_accuracy(embed_fn, dim_hint=None):
    doc_vecs = embed_fn(sanitized)
    index = faiss.IndexFlatIP(doc_vecs.shape[1])
    index.add(np.asarray(doc_vecs, dtype="float32"))
    hits = 0
    for q in queries:
        qv = embed_fn([sanitize(q["query"])])
        _, ids = index.search(np.asarray(qv, dtype="float32"), 3)
        hits += q["expected"] in ids[0]
    return hits / len(queries)

# Dynamic fallback checks to define models if they are missing from current runtime context
if 'embed' not in globals():
    print("⚠️ 'embed' function is missing in active namespace. Please run cell 's4tLspNS7PFL' to initialize Gemma.")
if 'embed_st' not in globals():
    try:
        print("🔄 Initializing SentenceTransformer inside this block as fallback...")
        from sentence_transformers import SentenceTransformer
        st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        def embed_st(texts):
            return st_model.encode(texts, normalize_embeddings=True)
    except Exception as e:
        print("⚠️ MiniLM fallback setup failed:", e)

if 'embed' in globals():
    gemma_acc = top3_accuracy(embed)
    print("Gemma 2 top-3:", gemma_acc)
else:
    print("❌ Skipping Gemma evaluation because model/embed is not ready.")

if 'embed_st' in globals() or 'embed_st' in locals():
    minilm_acc = top3_accuracy(embed_st)
    print("MiniLM top-3:", minilm_acc)
else:
    print("❌ Skipping MiniLM evaluation because it is not defined.")

In [ ]:
def show_failures(embed_fn, name):
    doc_vecs = embed_fn(sanitized)
    index = faiss.IndexFlatIP(doc_vecs.shape[1])
    index.add(np.asarray(doc_vecs, dtype="float32"))
    print(f"\n--- {name} ---")
    for q in queries:
        qv = embed_fn([sanitize(q["query"])])
        _, ids = index.search(np.asarray(qv, dtype="float32"), 3)
        if q["expected"] not in ids[0]:
            print("❌", q["query"], "| expected:", sanitized[q["expected"]])
            print("   top-3:", [sanitized[i] for i in ids[0]])

# Safely execute only if the embedding functions are defined in the workspace
if 'embed' in globals():
    show_failures(embed, "Gemma 2")
else:
    print("❌ 'embed' is not defined. Please run cell 's4tLspNS7PFL' to initialize Gemma first!")

if 'embed_st' in globals():
    show_failures(embed_st, "MiniLM")
else:
    print("❌ 'embed_st' is not defined. Please run cell 'l8lM5GvKFwPI' to initialize MiniLM first!")

In [ ]:
import json
import os

# Dynamically locate the correct JSON file path
paths = [
    "/content/data/labeled_sentences.json",
    "/content/gemma-embeddings-practice/data/labeled_sentences.json",
    "data/labeled_sentences.json"
]

target_file = None
for path in paths:
    if os.path.exists(path):
        target_file = path
        break

if target_file is None:
    # Default fallback to create it in the current directory if it does not exist anywhere
    target_file = "data/labeled_sentences.json"
    os.makedirs(os.path.dirname(target_file), exist_ok=True)
    print(f"⚠️ Target file not found. Creating a new dataset at {target_file}")
    data = {"documents": [], "queries": []}
else:
    print(f"📂 Found file at: {target_file}")
    with open(target_file, encoding="utf-8") as f:
        data = json.load(f)

# Update queries
data["queries"] = [q for q in data["queries"] if q["query"] != "kal ka weather kaisa rahega"]
data["queries"].append({"query": "aaj ka mausam kaisa hai", "expected": 3})

with open(target_file, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Queries:", len(data["queries"]))

In [ ]:
{"query": "aaj ka mausam kaisa hai", "expected": 3},

In [ ]:
import sys
import os
import shutil

# 1. Dynamically sync the 'app' module folder into current working directory if missing
current_dir = os.getcwd()
repo_app_path = "/content/gemma-embeddings-practice/app"
local_app_path = os.path.join(current_dir, "app")

if not os.path.exists(local_app_path):
    if os.path.exists(repo_app_path):
        shutil.copytree(repo_app_path, local_app_path)
        print(f"🔄 Synced 'app' directory from {repo_app_path} to {local_app_path}")
    elif os.path.exists("/content/app"):
        shutil.copytree("/content/app", local_app_path)
        print(f"🔄 Synced 'app' directory from /content/app to {local_app_path}")
    else:
         # Create a fallback detection.py if nothing is found
         os.makedirs(local_app_path, exist_ok=True)
         open(os.path.join(local_app_path, "__init__.py"), "w").close()
         with open(os.path.join(local_app_path, "detection.py"), "w") as f:
             f.write("import re\ndef sanitize(text: str) -> str:\n    return text\n")
         print("⚠️ 'app' folder not found anywhere. Created a dummy fallback fallback detection module.")

# Append paths to ensure import works
for path_to_add in ['/content/gemma-embeddings-practice', '/content', current_dir]:
    if path_to_add not in sys.path:
        sys.path.append(path_to_add)

# 2. Automatically install faiss-cpu if it's not present
try:
    import faiss
except ModuleNotFoundError:
    print("📦 'faiss' module missing. Installing 'faiss-cpu'...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
    import faiss

import json
import numpy as np
from app.detection import sanitize

# 3. Resolve correct file path or recreate dataset if completely missing
paths = [
    "/content/data/labeled_sentences.json",
    "/content/gemma-embeddings-practice/data/labeled_sentences.json",
    "data/labeled_sentences.json"
]
target_file = next((p for p in paths if os.path.exists(p)), None)

if not target_file:
    print("📂 Dataset file missing. Automatically reconstructing default labeled_sentences.json...")
    target_file = "data/labeled_sentences.json"
    os.makedirs(os.path.dirname(target_file), exist_ok=True)
    default_dataset = {
        "documents": [
            {"id": 0, "text": "Mera email jane@example.com hai", "pii": ["EMAIL"]},
            {"id": 1, "text": "Mujhe apna phone number 98765 43210 par call karo", "pii": ["PHONE"]},
            {"id": 2, "text": "Card 4111 1111 1111 1111 se payment hua", "pii": ["CREDIT_CARD"]},
            {"id": 3, "text": "Aaj mausam bahut achha hai", "pii": []},
            {"id": 4, "text": "Main cricket khelna pasand karta hoon", "pii": []},
            {"id": 5, "text": "Mera SSN 123-45-6789 hai", "pii": ["SSN"]},
            {"id": 6, "text": "Server ka IP 192.168.1.10 hai", "pii": ["IPV4"]},
            {"id": 7, "text": "Mera naya email priya.sharma@gmail.com hai", "pii": ["EMAIL"]},
            {"id": 8, "text": "Office ka number 011 4567 8901 hai", "pii": ["PHONE"]},
            {"id": 9, "text": "Hamara sales target is quarter mein badhaya gaya", "pii": []},
            {"id": 10, "text": "Naya laptop kharidna hai", "pii": []},
            {"id": 11, "text": "Credit card 5500 0000 0000 0004 se bill bhara", "pii": ["CREDIT_CARD"]},
            {"id": 12, "text": "Khana bahut tasty tha", "pii": []},
            {"id": 13, "text": "Meeting kal subah 10 baje hai", "pii": []},
            {"id": 14, "text": "Mera backup email rahul.k@yahoo.in hai", "pii": ["EMAIL"]},
            {"id": 15, "text": "Ghar ka router IP 10.0.0.1 hai", "pii": ["IPV4"]},
            {"id": 16, "text": "Dhoop aaj bahut tez hai", "pii": []},
            {"id": 17, "text": "Mujhe SSN update karna hai 987-65-4321", "pii": ["SSN"]},
            {"id": 18, "text": "Football match dekhna pasand hai", "pii": []},
            {"id": 19, "text": "Kal naya project shuru karna hai", "pii": []}
        ],
        "queries": [
            {"query": "kisi ka phone number kya hai", "expected": 1},
            {"query": "email bhejna hai", "expected": 0},
            {"query": "payment ka record kya hai", "expected": 2},
            {"query": "kal ka weather kaisa rahega", "expected": 3},
            {"query": "cricket ka match", "expected": 4},
            {"query": "server ka address kya hai", "expected": 6},
            {"query": "bank card ki details", "expected": 11},
            {"query": "sales ka target kya tha", "expected": 9},
            {"query": "football ke baare mein batao", "expected": 18},
            {"query": "kal ka plan kya hai", "expected": 19}
        ]
    }
    with open(target_file, "w", encoding="utf-8") as f:
        json.dump(default_dataset, f, ensure_ascii=False, indent=2)

with open(target_file, encoding="utf-8") as f:
    data = json.load(f)

docs = [d["text"] for d in data["documents"]]
sanitized = [sanitize(s) for s in docs]
queries = data["queries"]

def top3_accuracy(embed_fn):
    doc_vecs = embed_fn(sanitized)
    index = faiss.IndexFlatIP(doc_vecs.shape[1])
    index.add(np.asarray(doc_vecs, dtype="float32"))
    hits = 0
    for q in queries:
        qv = embed_fn([sanitize(q["query"])])
        _, ids = index.search(np.asarray(qv, dtype="float32"), 3)
        hits += q["expected"] in ids[0]
    return hits / len(queries)

# 4. Safely evaluate models if defined
if 'embed' in globals():
    print("Gemma 2 top-3:", top3_accuracy(embed))
else:
    print("❌ 'embed' (Gemma) is not defined in namespace. Run cell 's4tLspNS7PFL'.")

if 'embed_st' in globals():
    print("MiniLM top-3:", top3_accuracy(embed_st))
else:
    print("❌ 'embed_st' (MiniLM) is not defined in namespace. Run cell 'l8lM5GvKFwPI'.")

In [ ]:
pii_tests = [
    ("Mera email jane@example.com hai", [("EMAIL", "jane@example.com")]),
    ("Call 98765 43210 par karo", [("PHONE", "98765 43210")]),
    ("Mera SSN 123-45-6789 hai", [("SSN", "123-45-6789")]),
    ("Server IP 192.168.1.10 hai", [("IPV4", "192.168.1.10")]),
    ("Card 4111 1111 1111 1111 se payment", [("CREDIT_CARD", "4111 1111 1111 1111")]),
    ("Card 4111 1111 1111 1112 galat hai", []),          # invalid Luhn, PII nahi
    ("Aaj mausam achha hai", []),
    ("Main cricket khelta hoon", []),
    ("Mera backup email rahul.k@yahoo.in hai", [("EMAIL", "rahul.k@yahoo.in")]),
    ("Ghar ka router 10.0.0.1 hai", [("IPV4", "10.0.0.1")]),
    ("Meeting kal 10 baje hai", []),
    ("Office 011 4567 8901 par call karo", [("PHONE", "011 4567 8901")]),
]

In [ ]:
from app.detection import detect

tp = fp = fn = 0
for text, truth in pii_tests:
    gold = set()
    for label, value in truth:
        start = text.find(value)
        gold.add((label, start, start + len(value)))

    pred = {(l, s, e) for l, s, e in detect(text)}

    tp += len(pred & gold)
    fp += len(pred - gold)
    fn += len(gold - pred)

    if pred != gold:
        print("⚠️", text)
        print("   expected:", sorted(gold))
        print("   got     :", sorted(pred))

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
print(f"\nTP={tp} FP={fp} FN={fn}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

In [ ]:
p = "/content/app/detection.py"
s = open(p, encoding="utf-8").read()
old = '            if label == "CREDIT_CARD" and not luhn_valid(m.group(0)):\n                continue\n'
new = old + '            if label == "PHONE" and not 10 <= sum(c.isdigit() for c in m.group(0)) <= 15:\n                continue\n'
s = s.replace(old, new)
open(p, "w", encoding="utf-8").write(s)
print("patched" if 'label == "PHONE" and not 10' in s else "NOT patched, file check karo")

In [ ]:
!cp /content/app/detection.py /content/gemma-embeddings-practice/app/detection.py
%cd /content/gemma-embeddings-practice
!git add app/detection.py
!git commit -m "Fix: PHONE match only for 10-15 digits"
!git push origin main

In [ ]:
!cp /content/app/detection.py /content/gemma-embeddings-practice/app/detection.py
%cd /content/gemma-embeddings-practice
!git add app/detection.py
!git commit -m "Fix: PHONE match only for 10-15 digits"
!git push origin main

In [21]:
!grep -n 'label == "PHONE"' /content/app/detection.py

In [22]:
p = "/content/app/detection.py"
s = open(p, encoding="utf-8").read()
old = '            if label == "CREDIT_CARD" and not luhn_valid(m.group(0)):\n                continue\n'
new = old + '            if label == "PHONE" and not 10 <= sum(c.isdigit() for c in m.group(0)) <= 15:\n                continue\n'
s = s.replace(old, new)
open(p, "w", encoding="utf-8").write(s)
print("patched" if 'label == "PHONE" and not 10' in s else "NOT patched")

patched


In [23]:
!grep -n 'label == "PHONE"' /content/app/detection.py

30:            if label == "PHONE" and not 10 <= sum(c.isdigit() for c in m.group(0)) <= 15:


In [24]:
%cd /content/gemma-embeddings-practice
!cp /content/app/detection.py /content/gemma-embeddings-practice/app/detection.py
!git add app/detection.py
!git commit -m "Fix: PHONE match only for 10-15 digits"
!git push origin main

[Errno 2] No such file or directory: '/content/gemma-embeddings-practice'
/content
cp: cannot create regular file '/content/gemma-embeddings-practice/app/detection.py': No such file or directory
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [25]:
from google.colab import userdata
import os

GH_TOKEN = userdata.get("GITHUB_TOKEN")
USER = "Muskan-Savita"
REPO = "gemma-embeddings-practice"

%cd /content
!rm -rf gemma-embeddings-practice
!git clone https://{GH_TOKEN}@github.com/{USER}/{REPO}.git
!git config --global user.email "savitamuskan1404@gmail.com"
!git config --global user.name "Muskan-Savita"

/content
Cloning into 'gemma-embeddings-practice'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 39 (delta 7), reused 16 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 17.97 KiB | 5.99 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [26]:
%%writefile /content/gemma-embeddings-practice/app/detection.py
import re

RULES = {
    "CREDIT_CARD": re.compile(r"\b(?:\d[ -]?){12,18}\d\b"),
    "EMAIL": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "SSN":   re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "PHONE": re.compile(r"(?<!\d)\+?\d[\d\s-]{8,}\d(?!\d)"),
    "IPV4":  re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
}

def luhn_valid(number: str) -> bool:
    digits = [int(c) for c in number if c.isdigit()]
    if not 13 <= len(digits) <= 19:
        return False
    total = 0
    for i, d in enumerate(reversed(digits)):
        if i % 2 == 1:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0

def detect(text: str) -> list[tuple[str, int, int]]:
    out = []
    for label, pat in RULES.items():
        for m in pat.finditer(text):
            if label == "CREDIT_CARD" and not luhn_valid(m.group(0)):
                continue
            if label == "PHONE" and not 10 <= sum(c.isdigit() for c in m.group(0)) <= 15:
                continue
            out.append((label, m.start(), m.end()))

    out.sort(key=lambda x: (x[1], -(x[2] - x[1])))
    result, last_end = [], -1
    for f in out:
        if f[1] >= last_end:
            result.append(f)
            last_end = f[2]
    return result

def sanitize(text: str) -> str:
    for label, s, e in sorted(detect(text), key=lambda x: x[1], reverse=True):
        text = text[:s] + f"<{label}>" + text[e:]
    return text

Overwriting /content/gemma-embeddings-practice/app/detection.py


In [27]:
%cd /content/gemma-embeddings-practice
!git add app/detection.py
!git commit -m "Fix: PHONE match only for 10-15 digits"
!git push origin main

/content/gemma-embeddings-practice
[main c2a6925] Fix: PHONE match only for 10-15 digits
 1 file changed, 2 insertions(+)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 494 bytes | 494.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
fatal: could not read Username for 'https://github.com': No such device or address
send-pack: unexpected disconnect while reading sideband packet
error: failed to push some refs to 'https://github.com/Muskan-Savita/gemma-embeddings-practice.git'
fatal: the remote end hung up unexpectedly


In [28]:
%cd /content/gemma-embeddings-practice
import importlib, app.detection as d
importlib.reload(d)
print("PHONE check:", d.detect("Card 4111 1111 1111 1112 galat hai"))

/content/gemma-embeddings-practice
PHONE check: []


In [31]:
pii_tests = []
pii_tests += [
    ("Mera number +91 98765 43210 hai", [("PHONE", "+91 98765 43210")]),
    ("Ye number 9876543210 hai", [("PHONE", "9876543210")]),
    ("Order id 12345 hai, qty 3", []),
    ("Sale 2024-10-19 ko hui", []),
    ("Email: a.b+test@sub.example.co.in", [("EMAIL", "a.b+test@sub.example.co.in")]),
    ("Card 4111-1111-1111-1111 use kiya", [("CREDIT_CARD", "4111-1111-1111-1111")]),
    ("Ghar ka pin 110001 hai", []),
    ("Aaj 3 baje meeting hai", []),
]
print("Total test cases:", len(pii_tests))

Total test cases: 8


In [32]:
%cd /content/gemma-embeddings-practice
import importlib, app.detection as d
importlib.reload(d)
print("PHONE check:", d.detect("Card 4111 1111 1111 1112 galat hai"))

/content/gemma-embeddings-practice
PHONE check: []


In [33]:
pii_tests += [
    ("Mera number +91 98765 43210 hai", [("PHONE", "+91 98765 43210")]),
    ("Ye number 9876543210 hai", [("PHONE", "9876543210")]),
    ("Order id 12345 hai, qty 3", []),
    ("Sale 2024-10-19 ko hui", []),
    ("Email: a.b+test@sub.example.co.in", [("EMAIL", "a.b+test@sub.example.co.in")]),
    ("Card 4111-1111-1111-1111 use kiya", [("CREDIT_CARD", "4111-1111-1111-1111")]),
    ("Ghar ka pin 110001 hai", []),
    ("Aaj 3 baje meeting hai", []),
]
print("Total test cases:", len(pii_tests))

Total test cases: 16


In [34]:
from app.detection import detect

tp = fp = fn = 0
for text, truth in pii_tests:
    gold = set()
    for label, value in truth:
        start = text.find(value)
        gold.add((label, start, start + len(value)))

    pred = {(l, s, e) for l, s, e in detect(text)}

    tp += len(pred & gold)
    fp += len(pred - gold)
    fn += len(gold - pred)

    if pred != gold:
        print("⚠️", text)
        print("   expected:", sorted(gold))
        print("   got     :", sorted(pred))

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
print(f"\nTP={tp} FP={fp} FN={fn}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")


TP=8 FP=0 FN=0
Precision: 1.00
Recall: 1.00
